# Vinheria Agnello — Dashboard de Análise Histórica
## CP5 FIAP 2026 — Dados do STH-Comet

**Integrantes:**
- João Victor Melo (566640)
- Gustavo Macedo (567594)
- Gustavo Hiruo (567625)
- Yan Lucas (567046)

**Turma:** 1ESPA — Professor: Fábio Henrique Cabrini

---

Este notebook consome os dados históricos da entidade `urn:ngsi-ld:Vinheria:001` armazenados no STH-Comet (porta 8666) e gera gráficos com **auto-refresh a cada 5 segundos**.

**Importante:** Este notebook é um **complemento** do dashboard web em Python rodando como serviço Linux na porta 5000 (entrega principal do CP5).

## Importação de bibliotecas

In [ ]:
import requests
import matplotlib.pyplot as plt
from IPython.display import clear_output
from datetime import datetime
import time

## Configurações
IP da VM Azure onde o FIWARE está hospedado e ID da entidade.

In [ ]:
# IP da VM Azure com FIWARE
IP_VM = "20.124.178.183"
PORTA_STH = 8666

# Entidade no Orion/STH-Comet
ENTITY_ID = "urn:ngsi-ld:Vinheria:001"
ENTITY_TYPE = "SensorLDR"

# Headers padrão do FIWARE
HEADERS = {
    'fiware-service': 'smart',
    'fiware-servicepath': '/'
}

## Função de busca de dados no STH-Comet

In [ ]:
def obter_dados(atributo, lastN=20):
    """
    Busca os últimos N registros de um atributo no STH-Comet.
    Atributos disponíveis: luminosity, temperature, humidity
    """
    url = (
        f"http://{IP_VM}:{PORTA_STH}/STH/v1/contextEntities"
        f"/type/{ENTITY_TYPE}/id/{ENTITY_ID}"
        f"/attributes/{atributo}?lastN={lastN}"
    )
    try:
        response = requests.get(url, headers=HEADERS, timeout=5)
        if response.status_code == 200:
            data = response.json()
            valores = data['contextResponses'][0]['contextElement']['attributes'][0]['values']
            return valores
        else:
            print(f"Erro ao obter dados de {atributo}: {response.status_code}")
            return []
    except Exception as e:
        print(f"Erro de conexão ao buscar {atributo}: {e}")
        return []

## Função para plotar o dashboard
Gera 3 gráficos empilhados (luminosidade, temperatura, umidade) com linha de média.

In [ ]:
def plotar_dashboard(dados_lum, dados_temp, dados_hum):
    """
    Cria 3 gráficos empilhados com os dados dos 3 sensores e suas linhas de média.
    """
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    fig.suptitle('Vinheria Agnello — Monitoramento IoT em Tempo Real',
                 fontsize=16, fontweight='bold')

    sensores = [
        {'dados': dados_lum,  'titulo': 'Luminosidade (%)',  'cor': 'orange', 'ax': axes[0]},
        {'dados': dados_temp, 'titulo': 'Temperatura (°C)',  'cor': 'red',    'ax': axes[1]},
        {'dados': dados_hum,  'titulo': 'Umidade (%)',       'cor': 'blue',   'ax': axes[2]},
    ]

    for s in sensores:
        ax = s['ax']
        if not s['dados']:
            ax.text(0.5, 0.5, 'Aguardando dados do STH-Comet...',
                    ha='center', va='center', transform=ax.transAxes, color='gray')
            ax.set_title(s['titulo'])
            continue

        # Extrai valores e timestamps (mostra só HH:MM:SS)
        valores = [float(entry['attrValue']) for entry in s['dados']]
        tempos  = [entry['recvTime'][11:19] for entry in s['dados']]

        # Calcula média
        media = sum(valores) / len(valores)

        # Plot dos dados
        ax.plot(tempos, valores, marker='o', linestyle='-',
                color=s['cor'], linewidth=2)

        # Linha da média
        ax.axhline(media, color='blue', linestyle='--',
                   label=f'Média: {media:.2f}')

        ax.set_title(s['titulo'], fontweight='bold')
        ax.set_xlabel('Tempo')
        ax.set_ylabel('Valor')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right')

    plt.tight_layout()
    plt.show()

## Dashboard com auto-refresh

A célula abaixo executa um loop que:
1. Busca os 20 registros mais recentes do STH-Comet
2. Plota os 3 gráficos com linha de média
3. Aguarda 5 segundos
4. Limpa a tela e atualiza

Para parar a execução, clique no botão **stop** ao lado da célula.

**Observação:** Como o Colab tem latência maior que um servidor web local, o auto-refresh é mais lento que o dashboard principal rodando na VM (porta 5000).

In [ ]:
# Configurações do auto-refresh
INTERVALO_SEGUNDOS = 5   # tempo entre cada atualização
QTD_REGISTROS     = 20   # quantos registros históricos buscar

try:
    while True:
        # Busca os dados dos 3 sensores
        dados_lum  = obter_dados('luminosity',  QTD_REGISTROS)
        dados_temp = obter_dados('temperature', QTD_REGISTROS)
        dados_hum  = obter_dados('humidity',    QTD_REGISTROS)

        # Limpa a saída anterior e plota novamente
        clear_output(wait=True)
        agora = datetime.now().strftime('%d/%m/%Y %H:%M:%S')
        print(f'Última atualização: {agora}  |  Próxima em {INTERVALO_SEGUNDOS}s  |  Pressione STOP para parar')
        plotar_dashboard(dados_lum, dados_temp, dados_hum)

        # Aguarda antes da próxima atualização
        time.sleep(INTERVALO_SEGUNDOS)

except KeyboardInterrupt:
    print('\nAuto-refresh interrompido pelo usuário.')

## Versão estática (única execução)
Caso queira gerar um snapshot sem auto-refresh, execute a célula abaixo.

In [ ]:
dados_lum  = obter_dados('luminosity',  20)
dados_temp = obter_dados('temperature', 20)
dados_hum  = obter_dados('humidity',    20)

plotar_dashboard(dados_lum, dados_temp, dados_hum)